# Pesos de atenção do reference encoder — o que eles realmente medem

**Modelo:** Tacotron2-VAE com *pooling* por atenção (`verbo_v4_f1`, checkpoint `epoch_9500`) · **Corpus:** VERBO f1 (98 falas, 7 emoções)

## Por que as análises do site não fazem sentido

Todas as análises publicadas (sobreposição no eixo do tempo, "janelas de ~0,74 s", entropia como espalhamento temporal,
correlação com energia por janela de tempo) assumem que o índice $t$ de $\alpha_t$ percorre o **tempo**. Este notebook
mostra que essa premissa é **falsa** para os modelos treinados neste repositório:

O `ReferenceEncoder._cnn_forward` faz `inputs.contiguous().view(B, 1, -1, n_mels)` sobre um mel de layout **(B, 80, T)**.
`view` não transpõe — reinterpreta a memória. O resultado é que a "imagem" vista pelas convoluções não é (tempo × mel),
e cada passo da GRU acaba cobrindo uma **banda de canais mel ao longo da frase INTEIRA**, não uma janela de tempo:

- passo 0 → canais mel ≈ 0–19 (graves) · todos os instantes
- passo 1 → canais ≈ 20–39 · todos os instantes
- … · último passo → canais mais agudos

Ou seja: **α é um perfil de importância ESPECTRAL, não temporal.** Isso muda a leitura de tudo: o "viés de recência"
é peso nas bandas agudas; "tristeza concentrada no final" é tristeza dependente das bandas altas; e a correlação fraca
com energia-por-janela-de-tempo era esperada — eixo errado.

## O que este notebook faz

1. **Prova do layout** (Seção 1) — aritmética exata do `view` + campo receptivo por gradiente. Não depende de checkpoint.
2. **Perfil espectral de α** no corpus inteiro (Seção 3) — 98 falas, não 7.
3. **Validação causal por oclusão nos DOIS eixos** (Seção 4) — mascarar janelas de tempo vs. bandas mel e medir o efeito
   no embedding de estilo $\mu$. Se α mede influência, deve prever o efeito da oclusão de **bandas** e ignorar o de **tempo**.
4. **Saliência por gradiente** (Seção 5) — atribuição independente para confrontar com α.
5. **Emoções com n=98 e teste estatístico** (Seção 6) — perfis por emoção no eixo certo.

> Execute na máquina com o checkpoint (`experiments/tacotron2-vae/verbo_v4_f1/checkpoints/epoch_9500`), ou exporte
> `ATTN_CHECKPOINT=/caminho/para/checkpoint`. `ATTN_SMOKE_TEST=1` roda com pesos aleatórios (só para testar o código).

In [ ]:
import os, sys, json, math
from pathlib import Path
from collections import OrderedDict

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
ROOT = Path.cwd()
for p in ['src', 'src/models/tacotron2_vae', 'src/training/training-tacotron2-vae']:
    sys.path.insert(0, str(ROOT / p))

import numpy as np
import pandas as pd
import torch
import torchaudio
import librosa
import matplotlib.pyplot as plt
from scipy import stats

from layers import TacotronSTFT
from hparams import Tacotron2VAEHparams
from model import load_tacotron2_vae_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0); np.random.seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
                     'axes.spines.top': False, 'axes.spines.right': False})
MEL_FLOOR = float(np.log(1e-5))   # piso de silêncio do pipeline (dynamic_range_compression)
RESULTS = {}
print('device:', device)

In [ ]:
# ---------- carga do modelo (com descoberta de caminhos) ----------
CHECKPOINT = os.environ.get('ATTN_CHECKPOINT', '')
SMOKE = os.environ.get('ATTN_SMOKE_TEST') == '1'

EXP_CANDIDATES = [ROOT / 'experiments/tacotron2-vae/verbo_v4_f1',
                  ROOT / 'experiments/drive/tacotron2-vae/verbo_v4_f1']
DATA_CANDIDATES = [ROOT / 'data/raw/verbo/Audios/f1',
                   ROOT / 'data/raw/VERBO-Dataset/Audios/f1']

EXPERIMENT = next(p for p in EXP_CANDIDATES if (p / 'hparams.json').exists())
DATA_DIR = next(p for p in DATA_CANDIDATES if p.exists() and any(p.glob('*.wav')))

hparams = Tacotron2VAEHparams.from_dict(json.load(open(EXPERIMENT / 'hparams.json')))
model = load_tacotron2_vae_model(hparams, device=device)

ck_path = Path(CHECKPOINT) if CHECKPOINT else None
if ck_path is None:
    ckdir = EXPERIMENT / 'checkpoints'
    cks = sorted(ckdir.glob('epoch_*'), key=lambda p: int(p.name.split('_')[1])) if ckdir.exists() else []
    ck_path = cks[-1] if cks else None

if ck_path is not None and ck_path.exists():
    ck = torch.load(ck_path, map_location='cpu', weights_only=False)
    sd = OrderedDict((k[7:] if k.startswith('module.') else k, v)
                     for k, v in ck.get('state_dict', ck).items())
    assert any('.ref_encoder.attn.' in k for k in sd), (
        f'{ck_path} NAO tem o pooling por atencao (e um baseline); aponte ATTN_CHECKPOINT '
        'para o modelo com atencao, ex.: verbo_v4_f1/checkpoints/epoch_9500')
    model.load_state_dict(sd)
    print(f'checkpoint: {ck_path}  (iteration {ck.get("iteration", "?")})')
elif SMOKE:
    print('*' * 70)
    print('SMOKE TEST: PESOS ALEATORIOS - numeros sem significado, so valida o codigo')
    print('*' * 70)
else:
    raise FileNotFoundError('Checkpoint com atencao nao encontrado; exporte ATTN_CHECKPOINT '
                            'ou rode na maquina com verbo_v4_f1/checkpoints/epoch_9500.')

model.eval()
ref_enc = model.vae_gst.ref_encoder
N_MELS, HOP, SR = hparams.n_mel_channels, hparams.hop_length, hparams.sampling_rate
MEL_HZ = librosa.mel_frequencies(n_mels=N_MELS, fmin=hparams.mel_fmin, fmax=hparams.mel_fmax)
print(f'latente L={hparams.z_latent_dim} · GRU={hparams.ref_enc_gru_size} · dados: {DATA_DIR}')

## 1 · Prova do layout: o que cada passo da GRU cobre

O bloco nominal do passo $w$ são as linhas $[64w,\,64w{+}64)$ da imagem pós-`view`. Cada linha $r$ contém os índices
lineares $[80r,\,80r{+}80)$ da memória do mel (80, T), cujo índice linear é $p = m\cdot T + t$. Basta aritmética para
mapear cada célula (canal $m$, tempo $t$) ao seu passo — sem rodar o modelo.

In [ ]:
def step_of_cell(T, n_steps):
    '''Matriz (80, T) com o indice do passo da GRU que "possui" cada celula do mel.'''
    m_idx, t_idx = np.meshgrid(np.arange(N_MELS), np.arange(T), indexing='ij')
    flat = m_idx * T + t_idx                # indice linear na memoria (80, T)
    row = flat // N_MELS                    # linha da imagem pos-view
    return np.minimum(row // 64, n_steps - 1)

def n_steps_for(T):
    '''Altura de saida dos 6 convs stride 2 (k=3, pad=1): floor((H-1)/2)+1, 6x.'''
    h = T
    for _ in range(6):
        h = (h - 1) // 2 + 1
    return h

def band_ranges(T, n_steps):
    '''Faixa de canais mel [lo, hi] coberta pelo bloco nominal de cada passo.'''
    out = []
    for w in range(n_steps):
        lo = min(N_MELS - 1, (64 * w * N_MELS) // T)
        hi = min(N_MELS - 1, (64 * (w + 1) * N_MELS - 1) // T)
        out.append((lo, hi))
    return out

T_demo = 258                                    # fala tipica de ~3 s
ns = n_steps_for(T_demo)
owner = step_of_cell(T_demo, ns)

fig, ax = plt.subplots(figsize=(9, 3.2))
im = ax.imshow(owner, aspect='auto', origin='lower', cmap='tab10',
               extent=[0, T_demo * HOP / SR, 0, N_MELS])
ax.set(xlabel='tempo (s)', ylabel='canal mel',
       title=f'Dono de cada célula do mel (T={T_demo}, {ns} passos) — faixas HORIZONTAIS = bandas de frequência')
plt.colorbar(im, ax=ax, label='passo da GRU')
plt.tight_layout(); plt.show()

for w, (lo, hi) in enumerate(band_ranges(T_demo, ns)):
    print(f'passo {w}: canais mel {lo:2d}-{hi:2d}  ({MEL_HZ[lo]:5.0f}-{MEL_HZ[hi]:5.0f} Hz) — a frase INTEIRA no tempo')

Se α fosse temporal, o mapa acima teria faixas **verticais**. As faixas são **horizontais**: cada peso governa uma
banda de frequências da frase inteira. Confirmação com o modelo real (campo receptivo por gradiente — as convoluções
borram ±1 passo nas bordas, mas o suporte dominante é a banda):

In [ ]:
# gradiente da entrada da GRU no passo w em relacao ao mel -> suporte real (com overlap dos convs)
mel_probe = torch.randn(1, N_MELS, T_demo, device=device) * 0.1 + MEL_FLOOR
mel_probe.requires_grad_(True)
gru_in = ref_enc._cnn_forward(mel_probe)        # (1, T', 256)
ns_real = gru_in.size(1)
fig, axes = plt.subplots(1, min(3, ns_real), figsize=(13, 2.8), sharey=True)
for ax, w in zip(np.atleast_1d(axes), [0, ns_real // 2, ns_real - 1]):
    g = torch.autograd.grad(gru_in[0, w].pow(2).sum(), mel_probe, retain_graph=True)[0]
    ax.imshow(g[0].abs().cpu().numpy(), aspect='auto', origin='lower', cmap='magma')
    ax.set(title=f'|∂ passo {w} / ∂ mel|', xlabel='tempo (frames)')
np.atleast_1d(axes)[0].set_ylabel('canal mel')
plt.suptitle('Campo receptivo real de cada passo: faixas horizontais (bandas), não verticais (tempo)', y=1.05)
plt.tight_layout(); plt.show()
print(f'passos reais do conv stack para T={T_demo}: {ns_real} (formula: {n_steps_for(T_demo)})')

## 2 · Corpus e extração

Extraímos α e o embedding de estilo μ para as **98 falas** da locutora f1 (14 por emoção) e montamos uma tabela por
(fala, passo) com a banda mel correspondente. Como o nº de passos depende da duração ($T' = $ altura pós-convs), as
bandas têm resolução diferente por fala — por isso tudo adiante usa o **canal central da banda** como eixo comum.

In [ ]:
EMO = {'neu': 'neutro', 'ale': 'alegria', 'rai': 'raiva', 'tri': 'tristeza',
       'med': 'medo', 'sur': 'surpresa', 'des': 'desgosto'}
stft = TacotronSTFT(filter_length=hparams.filter_length, hop_length=HOP, win_length=hparams.win_length,
                    n_mel_channels=N_MELS, sampling_rate=SR, mel_fmin=hparams.mel_fmin, mel_fmax=hparams.mel_fmax)

utts, rows = [], []
MAX_UTTS = int(os.environ.get('ATTN_MAX_UTTS', '0')) or None
for wav_path in sorted(DATA_DIR.glob('*.wav'))[:MAX_UTTS]:
    codee = wav_path.name.split('-')[0]
    if codee not in EMO:
        continue
    wave, sr = torchaudio.load(str(wav_path))
    wave = wave.mean(dim=0, keepdim=True)
    if sr != SR:
        wave = torchaudio.functional.resample(wave, sr, SR)
    wave = wave.clamp(-1, 1)
    with torch.no_grad():
        mel = stft.mel_spectrogram(wave).to(device)                 # (1, 80, T)
        enc, alpha = ref_enc.encode_with_attention(mel)
        _, mu, _, _ = model.vae_gst._vae_from_enc(enc)
    a = alpha[0].cpu().numpy(); T = mel.size(2)
    bands = band_ranges(T, len(a))
    u = dict(utt=wav_path.stem, emotion=EMO[codee], mel=mel, alpha=a,
             mu=mu[0].cpu().numpy(), T=T, bands=bands, dur=wave.size(1) / SR)
    utts.append(u)
    for w, (aw, (lo, hi)) in enumerate(zip(a, bands)):
        band_e = mel[0, lo:hi + 1].mean().item()                    # energia log-mel media da banda
        rows.append(dict(utt=u['utt'], emotion=u['emotion'], w=w, n_w=len(a),
                         alpha=float(aw), ch_lo=lo, ch_hi=hi, ch_c=(lo + hi) / 2,
                         hz_c=float(MEL_HZ[(lo + hi) // 2]), band_energy=band_e))
df = pd.DataFrame(rows)
print(f'{len(utts)} falas · {len(df)} pares (fala, banda) · passos/fala: '
      f'{df.groupby("utt")["w"].max().add(1).min()}-{df.groupby("utt")["w"].max().add(1).max()}')
df.head()

## 3 · O perfil espectral da atenção

A pergunta corrigida: **quais bandas de frequência o encoder considera informativas para o estilo?**
E α é só um detector de energia por banda, ou carrega mais que isso?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
# perfil medio de alpha pelo canal central (todas as falas)
df['alpha_norm'] = df['alpha'] * df['n_w']            # 1.0 = uniforme, independe do n de passos
bins = pd.cut(df['ch_c'], bins=np.arange(0, 81, 8))
prof = df.groupby(bins, observed=True)['alpha_norm'].agg(['median', lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
x = [iv.mid for iv in prof.index]
axes[0].axhline(1.0, color='gray', ls='--', lw=1, label='uniforme')
axes[0].plot(x, prof['median'], 'o-', color='#2563eb', label='mediana')
axes[0].fill_between(x, prof.iloc[:, 1], prof.iloc[:, 2], alpha=.2, color='#2563eb')
axes[0].set(xlabel='canal mel central da banda', ylabel='α / uniforme',
            title='Perfil espectral médio da atenção (98 falas)')
sec = axes[0].secondary_xaxis('top', functions=(lambda c: np.interp(c, np.arange(80), MEL_HZ / 1000),
                                                lambda f: np.interp(f, MEL_HZ / 1000, np.arange(80))))
sec.set_xlabel('frequência (kHz)')
axes[0].legend()

# alpha vs energia da banda (correlacao no eixo CERTO, dentro de cada fala)
rho_e = df.groupby('utt').apply(lambda g: stats.spearmanr(g['alpha'], g['band_energy'])[0]
                                if g['w'].nunique() > 2 else np.nan, include_groups=False).dropna()
axes[1].hist(rho_e, bins=21, color='#2563eb', alpha=.8)
axes[1].axvline(rho_e.median(), color='crimson', label=f'mediana ρ = {rho_e.median():.2f}')
axes[1].set(xlabel='Spearman ρ(α, energia da banda) por fala', ylabel='nº de falas',
            title='α segue a energia espectral?')
axes[1].legend(); plt.tight_layout(); plt.show()
RESULTS['rho_alpha_energia_banda'] = float(rho_e.median())
print(f"mediana de ρ(α, energia da banda) = {rho_e.median():.3f}  "
      f"(Wilcoxon vs 0: p = {stats.wilcoxon(rho_e).pvalue:.2e})")

## 4 · Validação causal: oclusão nos dois eixos

O teste decisivo. Para cada fala mascaramos (piso de silêncio, `log(1e-5)`):

- **bandas**: os canais mel do bloco nominal de cada passo — a hipótese espectral;
- **tempo**: janelas de $T/T'$ frames — a interpretação (errada) do site;

e medimos $\Delta\mu = \lVert \mu_{\text{mascarado}} - \mu_{\text{completo}} \rVert$. Se α é uma medida de influência,
$\rho(\alpha, \Delta\mu_{\text{banda}})$ deve ser alto e $\rho(\alpha, \Delta\mu_{\text{tempo}})$ deve ser ~0.

In [ ]:
def mu_of(mel):
    with torch.no_grad():
        enc, _ = ref_enc.encode_with_attention(mel)
        _, mu, _, _ = model.vae_gst._vae_from_enc(enc)
    return mu[0].cpu().numpy()

rho_band, rho_time, pooled = [], [], []
for u in utts:
    mel, a, T, n = u['mel'], u['alpha'], u['T'], len(u['alpha'])
    if n <= 2:
        continue
    d_band, d_time = [], []
    tw = T / n
    for w, (lo, hi) in enumerate(u['bands']):
        mb = mel.clone(); mb[0, lo:hi + 1, :] = MEL_FLOOR            # oclui a BANDA
        d_band.append(np.linalg.norm(mu_of(mb) - u['mu']))
        t0, t1 = int(w * tw), int((w + 1) * tw)
        mt = mel.clone(); mt[0, :, t0:t1] = MEL_FLOOR                # oclui a JANELA DE TEMPO
        d_time.append(np.linalg.norm(mu_of(mt) - u['mu']))
    rho_band.append(stats.spearmanr(a, d_band)[0])
    rho_time.append(stats.spearmanr(a, d_time)[0])
    db = np.array(d_band)
    pooled += [(ai, di / db.max()) for ai, di in zip(a * n, db)]

rho_band, rho_time = np.array(rho_band), np.array(rho_time)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].boxplot([rho_band, rho_time], tick_labels=['oclusão de BANDA', 'oclusão de TEMPO'], widths=.5)
axes[0].axhline(0, color='gray', lw=1)
axes[0].set(ylabel='Spearman ρ(α, Δμ) por fala', title='α prevê o efeito causal de qual oclusão?')
px, py = np.array(pooled).T
axes[1].scatter(px, py, s=8, alpha=.35, color='#2563eb')
axes[1].set(xlabel='α / uniforme', ylabel='Δμ banda (normalizado por fala)',
            title=f'Pooled: ρ = {stats.spearmanr(px, py)[0]:.2f}')
plt.tight_layout(); plt.show()

RESULTS['rho_occl_banda'] = float(np.median(rho_band))
RESULTS['rho_occl_tempo'] = float(np.median(rho_time))
print(f'mediana ρ(α, Δμ_banda) = {np.median(rho_band):.3f}   '
      f'mediana ρ(α, Δμ_tempo) = {np.median(rho_time):.3f}')
print(f'Wilcoxon pareado banda vs tempo: p = {stats.wilcoxon(rho_band, rho_time).pvalue:.2e}')

**Como ler:** ρ alto na oclusão de banda = α é um mapa de atribuição *validado causalmente* — remover a banda que
ele aponta muda o estilo proporcionalmente ao peso. ρ ~0 na oclusão de tempo confirma que ler α como linha do tempo
(como o site faz) não tem suporte causal.

## 5 · Saliência por gradiente: atribuição independente

$\lVert \partial \mu / \partial \text{mel} \rVert$ agregada por banda é um segundo mapa de importância, calculado sem
usar α. Se os dois concordam, a leitura espectral fica ainda mais sólida.

In [ ]:
def saliency(mel):
    m = mel.clone().requires_grad_(True)
    with torch.backends.cudnn.flags(enabled=False):   # backward de RNN em eval exige cudnn off
        enc, _ = ref_enc.encode_with_attention(m)
        _, mu, _, _ = model.vae_gst._vae_from_enc(enc)
        mu.pow(2).sum().backward()
    return m.grad[0].abs().cpu().numpy()          # (80, T)

rho_grad = []
show = [u for e in ['neutro', 'tristeza', 'raiva'] for u in utts if u['emotion'] == e][:3]
fig, axes = plt.subplots(1, len(show), figsize=(13, 3.0))
for u in utts:
    sal = saliency(u['mel'])
    g_band = [sal[lo:hi + 1].mean() for lo, hi in u['bands']]
    if len(g_band) > 2:
        rho_grad.append(stats.spearmanr(u['alpha'], g_band)[0])
    if u in show:
        ax = np.atleast_1d(axes)[show.index(u)]
        ax.imshow(sal, aspect='auto', origin='lower', cmap='magma')
        for lo, hi in u['bands'][:-1]:
            ax.axhline(hi + .5, color='cyan', lw=.6, ls=':')
        ax.set(title=f"{u['utt']} ({u['emotion']})", xlabel='tempo (frames)')
np.atleast_1d(axes)[0].set_ylabel('canal mel')
plt.suptitle('|∂μ/∂mel|: a saliência se organiza em FAIXAS HORIZONTAIS (bandas, linhas ciano = fronteiras dos passos)', y=1.06)
plt.tight_layout(); plt.show()

rho_grad = np.array(rho_grad)
RESULTS['rho_grad'] = float(np.median(rho_grad))
print(f'mediana ρ(α, saliência da banda) = {np.median(rho_grad):.3f}  '
      f'(Wilcoxon vs 0: p = {stats.wilcoxon(rho_grad).pvalue:.2e})')

## 6 · Emoções, com o corpus inteiro e o eixo certo

Reinterpretação do achado do site ("tristeza concentra atenção no final") com n=98: perfis espectrais de α por emoção.
"Final" = **bandas agudas**. Também comparamos com o perfil de energia por banda, para separar "α segue a emoção"
de "α segue a energia, que muda com a emoção".

In [ ]:
# projeta alpha de cada fala numa grade fixa de 80 canais
def to_grid(u, values):
    g = np.zeros(N_MELS)
    for (lo, hi), v in zip(u['bands'], values):
        g[lo:hi + 1] = v
    return g

grid_a = {e: [] for e in EMO.values()}
ent = []
for u in utts:
    n = len(u['alpha'])
    grid_a[u['emotion']].append(to_grid(u, u['alpha'] * n))
    if n > 1:
        ent.append(dict(emotion=u['emotion'],
                        H=float(-(u['alpha'] * np.log(u['alpha'] + 1e-12)).sum() / np.log(n))))
ent = pd.DataFrame(ent)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
cmap = plt.get_cmap('tab10')
for i, (e, curves) in enumerate(grid_a.items()):
    axes[0].plot(np.arange(N_MELS), np.mean(curves, axis=0), color=cmap(i), label=e, lw=1.8)
axes[0].axhline(1.0, color='gray', ls='--', lw=1)
axes[0].set(xlabel='canal mel', ylabel='α / uniforme', title='Perfil espectral médio de α por emoção (n=14 cada)')
axes[0].legend(ncol=2, fontsize=8)

order = ent.groupby('emotion')['H'].median().sort_values().index
axes[1].boxplot([ent.loc[ent.emotion == e, 'H'] for e in order], tick_labels=list(order), widths=.6)
kw = stats.kruskal(*[ent.loc[ent.emotion == e, 'H'] for e in order])
axes[1].set(ylabel='entropia normalizada de α', title=f'Entropia por emoção — Kruskal-Wallis p = {kw.pvalue:.3g}')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
RESULTS['kruskal_entropia_p'] = float(kw.pvalue)
print(ent.groupby('emotion')['H'].agg(['median', 'std']).round(3).sort_values('median'))

## 7 · Conclusões

**O sentido útil dos pesos de atenção do reference encoder é espectral, não temporal:**

1. **Layout (Seção 1, fato arquitetural):** o `view` sem transpose faz cada passo da GRU cobrir uma banda de canais mel
   da frase inteira. Toda leitura temporal de α (site: "janelas de 0,74 s", "viés de recência", "atenção no final da
   frase") está no eixo errado. O "viés de recência" é, na verdade, peso nas **bandas agudas**.
2. **α como atribuição espectral validada causalmente (Seções 4–5):** ocluir a banda apontada por α move o embedding μ
   proporcionalmente ao peso (ρ mediano alto), ocluir janelas de tempo não tem relação com α, e a saliência por
   gradiente concorda com α. Ou seja: **α diz quais faixas de frequência da referência constroem o vetor de estilo** —
   um "equalizador de importância" sobre o timbre/registro da locutora.
3. **Emoções (Seção 6):** os perfis por emoção devem ser lidos como assinaturas espectrais (ex.: emoções de alta
   ativação espalham importância pelas bandas; baixa ativação concentra nas bandas específicas), com n=98 e teste
   estatístico — não 7 números.

**Consequências práticas:**
- O texto do site/relatório sobre "janelas de ~0,74 s" e as figuras com α sobreposto ao eixo do tempo precisam ser
  reescritos como perfis espectrais (α vs. canal mel/Hz).
- Se a intenção original era atenção *temporal* de verdade, o conserto é uma linha —
  `inputs.transpose(1, 2).contiguous().view(B, 1, -1, n_mels)` em `ReferenceEncoder._cnn_forward` — mas exige
  **retreinar** os modelos; os checkpoints atuais devem ser interpretados como atenção espectral.
- A interpretabilidade continua real e até mais interessante: o modelo aprendeu sozinho a ponderar bandas de
  frequência para extrair estilo, e a ponderação é causalmente fiel.

In [ ]:
print('=' * 64)
print('RESUMO AUTOMATICO')
print('=' * 64)
labels = {
    'rho_alpha_energia_banda': 'ρ mediano (α, energia da banda)          ',
    'rho_occl_banda':          'ρ mediano (α, Δμ oclusão de BANDA)  [alto = α é causal]',
    'rho_occl_tempo':          'ρ mediano (α, Δμ oclusão de TEMPO)  [~0 = leitura temporal invalida]',
    'rho_grad':                'ρ mediano (α, saliência por gradiente)   ',
    'kruskal_entropia_p':      'Kruskal-Wallis entropia×emoção (p)       ',
}
for k, v in RESULTS.items():
    print(f'  {labels.get(k, k)} = {v:.4g}')
if os.environ.get('ATTN_SMOKE_TEST') == '1':
    print('\n  [SMOKE TEST: pesos aleatorios -> estes numeros NAO significam nada]')